# Study 951 — The Crossover Rung — the teardown

Four rungs (AGG, LQD, ANGL, HYG) on daily total-return closes, excess of BIL, regressed jointly on IEF (duration) and SPY (equity beta). The headline statistic is the two-factor alpha on the **return difference** between rungs, with a Newey-West HAC *t*. Then: a joint block-bootstrap CI, an era cut, a one-year-out jackknife, an HAC-bandwidth sweep, the sibling-fund pair, the borrow sweep, and the live synthetic control.

Every real number is frozen from `docs/results.md` (Fingerprint `b2e46afc7601`), window 2012-04-11 → 2026-06-30, n = 3,574 regression days. **Proxies:** fallen-angel ETFs stand in for the crossover rung (no free daily BBB−/BB+ tape); IEF/SPY are a two-factor adjustment, not a credit-factor model, and their betas are **fitted in sample** (full-window OLS, in-sample intercept — standard attribution, and the fitted hedge is never traded); the short-leg borrow rate is an assumption and is swept.

In [1]:
R = {'start': '2012-04-11', 'end': '2026-06-30', 'n_days': 3575, 'n_reg': 3574, 'fp': 'b2e46afc7601', 'agg': (0.113, 0.55, 4.86, -18.4, -0.48, -0.94, 0.71, 0.07, 0.84), 'lqd': (0.241, 1.89, 7.82, -25.0, -0.67, -0.49, 0.97, 0.18, 0.67), 'angl': (0.602, 5.57, 9.25, -29.3, 0.84, 0.42, 0.34, 0.35, 0.39), 'hyg': (0.437, 3.42, 7.82, -22.0, -1.67, -1.23, 0.27, 0.37, 0.62), 'ah_alpha': 2.51, 'ah_t': 2.01, 'ah_raw': 2.15, 'ah_raw_t': 1.96, 'al_alpha': 1.51, 'al_t': 1.15, 'al_raw': 3.68, 'al_raw_t': 2.09, 'hl_alpha': -1.0, 'hl_t': -0.83, 'hl_raw': 1.53, 'hl_raw_t': 0.92, 'hl_long_alpha': 0.05, 'hl_long_t': 0.03, 'hl_long_n': 4801, 'ah_ci': (0.38, 4.74), 'ah_neg': 1.3, 'al_ci': (-0.61, 3.9), 'al_neg': 8.0, 'era_e_n': 1691, 'era_e_ah': 4.47, 'era_e_ah_t': 2.63, 'era_e_al': 3.94, 'era_e_al_t': 2.02, 'era_l_n': 1882, 'era_l_ah': 1.5, 'era_l_ah_t': 0.92, 'era_l_al': -0.05, 'era_l_al_t': -0.03, 'cal': [(2012, 2.12), (2013, 1.15), (2014, 0.63), (2015, 3.6), (2016, 10.65), (2017, 3.41), (2018, -3.92), (2019, 3.48), (2020, 8.48), (2021, 2.98), (2022, -3.74), (2023, 0.82), (2024, -1.77), (2025, 0.47), (2026, 0.58)], 'jk_pass': 5, 'jk_n': 15, 'jk_drop16': (1.84, 1.43), 'jk_drop20': (2.32, 2.42), 'drop_both': (1.55, 1.61), 'drop_both_n': 3069, 'lags': [(5, 1.95), (8, 2.01), (10, 2.01), (21, 2.25), (42, 2.54)], 'faln_ushy': (0.63, 0.48, 2178), 'faln_hyg': (1.78, 1.34, 2519), 'angl_ushy': (0.18, 0.14, 2178), 'same_win': [('ANGL - HYG (headline)', 0.96, 0.67), ('FALN - HYG', 1.42, 0.97), ('FALN - USHY', 0.63, 0.48), ('ANGL - USHY', 0.18, 0.14)], 'same_win_n': 2178, 'late_ladder': [('LQD', 0.041, -1.12), ('FALN', 0.364, -1.01), ('ANGL', 0.312, -1.46), ('USHY', 0.311, -1.64), ('HYG', 0.244, -2.42)], 'swap_gain': 0.165, 'borrow': [(0, 2.53, 2.11, 2.14, 0.342), (25, 2.28, 1.9, 1.89, 0.302), (50, 2.03, 1.69, 1.64, 0.262), (100, 1.53, 1.28, 1.14, 0.182)], 'turnover': 0.38, 'syn_planted': 3.0, 'syn_recovered': 3.58, 'syn_t': 4.64, 'syn_null_mean': 0.1, 'syn_null_sd': 0.6, 'syn_null_fire': 0, 'syn_null_n': 8}

## 1. The ladder — levels and loadings

Columns: excess-of-cash Sharpe, annualised excess return, annualised vol, *absolute* max drawdown, two-factor alpha and its HAC *t*, the two betas, R².

> 💡 **In plain words** — the first column is what each rung paid per unit of wobble; the alpha column is what is left after you hold the matching amount of Treasuries and stocks against it.

In [2]:
hdr = '%-6s %9s %10s %8s %9s %11s %8s %7s %7s %6s'
print(hdr % ('rung','exSharpe','excess/yr','vol','maxDD','alpha/yr','t','bIEF','bSPY','R2'))
for k, nm in [('agg','AGG'), ('lqd','LQD'), ('angl','ANGL'), ('hyg','HYG')]:
    s, exr, vol, dd, a, t, bd, be, r2 = R[k]
    print('%-6s %+9.3f %+9.2f%% %7.2f%% %+8.1f%% %+10.2f%% %+8.2f %7.2f %7.2f %6.2f'
          % (nm, s, exr, vol, dd, a, t, bd, be, r2))
print()
print('The raw excess-Sharpe ladder humps on the crossover rung (%.3f) --' % R['angl'][0])
print('but no single rung alpha clears |t| = 2 on its own.')

rung    exSharpe  excess/yr      vol     maxDD    alpha/yr        t    bIEF    bSPY     R2
AGG       +0.113     +0.55%    4.86%    -18.4%      -0.48%    -0.94    0.71    0.07   0.84
LQD       +0.241     +1.89%    7.82%    -25.0%      -0.67%    -0.49    0.97    0.18   0.67
ANGL      +0.602     +5.57%    9.25%    -29.3%      +0.84%    +0.42    0.34    0.35   0.39
HYG       +0.437     +3.42%    7.82%    -22.0%      -1.67%    -1.23    0.27    0.37   0.62

The raw excess-Sharpe ladder humps on the crossover rung (0.602) --
but no single rung alpha clears |t| = 2 on its own.


## 2. Head-to-head — alpha on the return difference

Both legs are excess-of-cash, so the cash leg cancels in the difference and the intercept is exactly the duration- and equity-adjusted premium of one rung over another (the Jobson-Korkie return-difference form, HAC standard errors). All three pairs run on the **same ANGL-gated window** so the rows are comparable; HYG − LQD is the one pair that does not need ANGL, and its own longest (BIL-gated, n = 4,801) window is printed underneath, labelled, rather than mixed into the table.

In [3]:
rows = [('ANGL - HYG (boundary vs deep HY)', R['ah_alpha'], R['ah_t'], R['ah_raw'], R['ah_raw_t']),
        ('ANGL - LQD (boundary vs IG)     ', R['al_alpha'], R['al_t'], R['al_raw'], R['al_raw_t']),
        ('HYG  - LQD (deep HY vs IG)      ', R['hl_alpha'], R['hl_t'], R['hl_raw'], R['hl_raw_t'])]
print('%-34s %10s %7s %10s %7s' % ('pair','alpha/yr','HAC t','raw/yr','raw t'))
for nm, a, t, raw, rt in rows:
    print('%-34s %+9.2f%% %+7.2f %+9.2f%% %+7.2f' % (nm, a, t, raw, rt))
print()
print('  (all three rows: same ANGL-gated window, n = %d)' % R['n_reg'])
print()
print('ANGL - LQD: the adjustment removes %.0f%% of the raw difference'
      % ((1 - R['al_alpha']/R['al_raw'])*100))
print('HYG  - LQD: the IG->HY step paid %+.2f%%/yr adjusted -- nothing at all' % R['hl_alpha'])
print('            on its own longest window (n = %d) it reads %+.2f%%/yr (t = %+.2f)'
      % (R['hl_long_n'], R['hl_long_alpha'], R['hl_long_t']))

pair                                 alpha/yr   HAC t     raw/yr   raw t
ANGL - HYG (boundary vs deep HY)       +2.51%   +2.01     +2.15%   +1.96
ANGL - LQD (boundary vs IG)            +1.51%   +1.15     +3.68%   +2.09
HYG  - LQD (deep HY vs IG)             -1.00%   -0.83     +1.53%   +0.92

  (all three rows: same ANGL-gated window, n = 3574)

ANGL - LQD: the adjustment removes 59% of the raw difference
HYG  - LQD: the IG->HY step paid -1.00%/yr adjusted -- nothing at all
            on its own longest window (n = 4801) it reads +0.05%/yr (t = +0.03)


## 3. Block-bootstrap CI (2,000 draws, 21-day blocks)

Blocks are resampled **jointly** across the difference series and the factor tape, so the betas are re-estimated on every draw and the interval carries their estimation error, not just the intercept's.

In [4]:
print('ANGL - HYG : alpha %+.2f%%  95%% CI [%+.2f%%, %+.2f%%]  share<0 %.1f%%'
      % (R['ah_alpha'], R['ah_ci'][0], R['ah_ci'][1], R['ah_neg']))
print('ANGL - LQD : alpha %+.2f%%  95%% CI [%+.2f%%, %+.2f%%]  share<0 %.1f%%'
      % (R['al_alpha'], R['al_ci'][0], R['al_ci'][1], R['al_neg']))
print()
print('The HY leg clears zero with %.2f pp to spare at the lower bound.' % R['ah_ci'][0])
print('The IG leg does not clear zero.')

ANGL - HYG : alpha +2.51%  95% CI [+0.38%, +4.74%]  share<0 1.3%
ANGL - LQD : alpha +1.51%  95% CI [-0.61%, +3.90%]  share<0 8.0%

The HY leg clears zero with 0.38 pp to spare at the lower bound.
The IG leg does not clear zero.


## 4. Era cut (split 2019-01-01)

> 💡 **In plain words** — the premium is a first-half phenomenon by a factor of three, and the second half is *not* a quiet period for the mechanism: it contains the March-2020 fallen-angel wave, the largest downgrade cohort on record.

In [5]:
print('%-12s %6s %12s %8s' % ('pair','n','alpha/yr','HAC t'))
print('%-12s %6d %+11.2f%% %+8.2f' % ('ANGL-HYG e', R['era_e_n'], R['era_e_ah'], R['era_e_ah_t']))
print('%-12s %6d %+11.2f%% %+8.2f' % ('ANGL-HYG l', R['era_l_n'], R['era_l_ah'], R['era_l_ah_t']))
print('%-12s %6d %+11.2f%% %+8.2f' % ('ANGL-LQD e', R['era_e_n'], R['era_e_al'], R['era_e_al_t']))
print('%-12s %6d %+11.2f%% %+8.2f' % ('ANGL-LQD l', R['era_l_n'], R['era_l_al'], R['era_l_al_t']))
print()
print('ANGL-HYG decays %.1fx across the split; ANGL-LQD goes to zero.'
      % (R['era_e_ah'] / R['era_l_ah']))

pair              n     alpha/yr    HAC t
ANGL-HYG e     1691       +4.47%    +2.63
ANGL-HYG l     1882       +1.50%    +0.92
ANGL-LQD e     1691       +3.94%    +2.02
ANGL-LQD l     1882       -0.05%    -0.03

ANGL-HYG decays 3.0x across the split; ANGL-LQD goes to zero.


## 5. Influence — calendar years and the one-year-out jackknife

The forced-seller mechanism predicts an *episodic* payoff, and that is exactly what the tape shows — which is also what destroys the significance.

In [6]:
tot = sum(v for _, v in R['cal'])
waves = sum(v for y, v in R['cal'] if y in (2016, 2020))
for y, v in R['cal']:
    print('%d %+6.2f%%%s' % (y, v, '   <- downgrade wave' if y in (2016, 2020) else ''))
print()
print('2016 + 2020 alone = %+.2f pp of the %+.2f pp cumulative sum (%.0f%%)'
      % (waves, tot, 100*waves/tot))
print('|t| >= 2 survives %d/%d single-year deletions' % (R['jk_pass'], R['jk_n']))
print('drop 2016      : alpha %+.2f%% (t = %+.2f)' % R['jk_drop16'])
print('drop 2020      : alpha %+.2f%% (t = %+.2f)' % R['jk_drop20'])
print('drop both      : alpha %+.2f%% (t = %+.2f)  over %d days'
      % (R['drop_both'][0], R['drop_both'][1], R['drop_both_n']))

2012  +2.12%
2013  +1.15%
2014  +0.63%
2015  +3.60%
2016 +10.65%   <- downgrade wave
2017  +3.41%
2018  -3.92%
2019  +3.48%
2020  +8.48%   <- downgrade wave
2021  +2.98%
2022  -3.74%
2023  +0.82%
2024  -1.77%
2025  +0.47%
2026  +0.58%

2016 + 2020 alone = +19.13 pp of the +28.94 pp cumulative sum (66%)
|t| >= 2 survives 5/15 single-year deletions
drop 2016      : alpha +1.84% (t = +1.43)
drop 2020      : alpha +2.32% (t = +2.42)
drop both      : alpha +1.55% (t = +1.61)  over 3069 days


## 6. HAC bandwidth and the sibling-fund pair

The bandwidth sweep runs in the *unhelpful* direction — the *t* is lowest at short bandwidths and rises with the kernel — so the headline (default `4(n/100)^(2/9)` = 8 lags) is not a cherry-picked wide window. The fund swap looks like the harder test — replace **both** legs with their siblings and the premium is gone — but the sibling funds only exist from 2017, so the last block re-runs all four pairs on the one window they share. The headline pair collapses there too: the swap is mostly the era cut in disguise.

In [7]:
for lg, t in R['lags']:
    print('HAC lags %3d : t = %+.2f%s' % (lg, t, '   <- default rule of thumb' if lg == 8 else ''))
print()
print('%-28s %10s %8s %7s' % ('pair','alpha/yr','HAC t','n'))
print('%-28s %+9.2f%% %+8.2f %7d' % ('ANGL - HYG  (headline)', R['ah_alpha'], R['ah_t'], R['n_reg']))
for nm, key in [('FALN - USHY (both swapped)','faln_ushy'),
                ('FALN - HYG  (cross leg)','faln_hyg'),
                ('ANGL - USHY (HY leg)','angl_ushy')]:
    a, t, n = R[key]
    print('%-28s %+9.2f%% %+8.2f %7d' % (nm, a, t, n))
print()
print('ladder on the FALN-gated 2017-2026 window (excess Sharpe, alpha):')
for nm, s, a in R['late_ladder']:
    print('  %-5s %+.3f  %+.2f%%' % (nm, s, a))
print('  the hump survives; every alpha is negative and every |t| < 1.4')
print()
print('window-vs-fund: all four pairs on the shared window (n = %d)' % R['same_win_n'])
for nm, a, t in R['same_win']:
    print('  %-22s %+6.2f%%   t %+5.2f' % (nm, a, t))
print('  the HEADLINE pair loses %.2f pp on this window alone --'
      % (R['ah_alpha'] - R['same_win'][0][1]))
print('  so the sibling swap is mostly the era cut restated, not a third failure')

HAC lags   5 : t = +1.95
HAC lags   8 : t = +2.01   <- default rule of thumb
HAC lags  10 : t = +2.01
HAC lags  21 : t = +2.25
HAC lags  42 : t = +2.54

pair                           alpha/yr    HAC t       n
ANGL - HYG  (headline)           +2.51%    +2.01    3574
FALN - USHY (both swapped)       +0.63%    +0.48    2178
FALN - HYG  (cross leg)          +1.78%    +1.34    2519
ANGL - USHY (HY leg)             +0.18%    +0.14    2178

ladder on the FALN-gated 2017-2026 window (excess Sharpe, alpha):
  LQD   +0.041  -1.12%
  FALN  +0.364  -1.01%
  ANGL  +0.312  -1.46%
  USHY  +0.311  -1.64%
  HYG   +0.244  -2.42%
  the hump survives; every alpha is negative and every |t| < 1.4

window-vs-fund: all four pairs on the shared window (n = 2178)
  ANGL - HYG (headline)   +0.96%   t +0.67
  FALN - HYG              +1.42%   t +0.97
  FALN - USHY             +0.63%   t +0.48
  ANGL - USHY             +0.18%   t +0.14
  the HEADLINE pair loses 1.55 pp on this window alone --
  so the sibling swap

## 7. The tradable expressions — costs, borrow, and one execution lag

**A.** Own ANGL instead of HYG: no shorting, no timing, a single switch charged on **both** sides (sell HYG *and* buy ANGL, 2 × 5 bps one-way) and amortised over 14 years — ~0.007%/yr of drag — fund fees already inside the total-return tape.

**B.** The pure spread (long ANGL, short HYG), dollar-neutral, reset monthly: the weights implied by the closes through the last trading day of month *t* are acted on at *t*+1 — **the study's single execution lag** — where the drift turnover (0.38× per year) is charged at 5 bps one-way × NAV. The short leg pays borrow, which is an **assumption**, so it is swept.

In [8]:
print('A. own ANGL instead of HYG: excess-Sharpe gain %+.3f, drawdown %.1f%% -> %.1f%%'
      % (R['swap_gain'], R['hyg'][3], R['angl'][3]))
print()
print('B. long ANGL / short HYG, monthly reset, turnover %.2fx/yr' % R['turnover'])
print('%8s %10s %8s %9s %8s' % ('borrow','alpha/yr','HAC t','mean/yr','Sharpe'))
for b, a, t, m, s in R['borrow']:
    flag = '   <- realistic ETF borrow' if b == 50 else ''
    print('%6d bp %+9.2f%% %+8.2f %+8.2f%% %+8.3f%s' % (b, a, t, m, s, flag))

A. own ANGL instead of HYG: excess-Sharpe gain +0.165, drawdown -22.0% -> -29.3%

B. long ANGL / short HYG, monthly reset, turnover 0.38x/yr
  borrow   alpha/yr    HAC t   mean/yr   Sharpe
     0 bp     +2.53%    +2.11    +2.14%   +0.342
    25 bp     +2.28%    +1.90    +1.89%   +0.302
    50 bp     +2.03%    +1.69    +1.64%   +0.262   <- realistic ETF borrow
   100 bp     +1.53%    +1.28    +1.14%   +0.182


## 8. Synthetic control — the estimator is unbiased

A three-rung ladder built from a duration factor and an equity factor, with a planted premium on the crossover rung. It must be recovered when present and absent when not. **Synthetic only — this never supports a real-tape stamp.**

In [9]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))
sys.path.insert(0, os.path.abspath(os.path.join('..','..','..')))
import numpy as np
from crossover_credit import data, strategy as st
planted, truth = data.synthetic_panel(signal_strength=1.0, seed=951)
d = st.synthetic_detect(planted)
print('planted %+.2f%%/yr -> recovered %+.2f%%/yr (t = %+.2f)'
      % (truth['alpha_planted']*100, d['alpha_ann']*100, d['t_alpha']))
print('recovered betas: dur %.2f, eq %.2f  (planted rung is dur 0.35 / eq 0.35 vs HY 0.25 / 0.40)'
      % (d['betas']['dur'], d['betas']['eq']))
nulls = [st.synthetic_detect(data.synthetic_panel(signal_strength=0.0, seed=951+s)[0])
         for s in range(8)]
a = np.array([x['alpha_ann'] for x in nulls]) * 100
t = np.array([x['t_alpha'] for x in nulls])
print('null x8: alpha mean %+.2f%%/yr (sd %.2f), |t| >= 2 fires %d/8'
      % (a.mean(), a.std(ddof=1), (abs(t) >= 2).sum()))
half = st.synthetic_detect(data.synthetic_panel(signal_strength=0.5, seed=951)[0])
print('half strength: %+.2f%%/yr (t = %+.2f) -- the knob is monotone'
      % (half['alpha_ann']*100, half['t_alpha']))

planted +3.00%/yr -> recovered +3.58%/yr (t = +4.64)
recovered betas: dur 0.10, eq -0.06  (planted rung is dur 0.35 / eq 0.35 vs HY 0.25 / 0.40)


null x8: alpha mean +0.10%/yr (sd 0.60), |t| >= 2 fires 0/8


half strength: +2.08%/yr (t = +2.69) -- the knob is monotone


## Verdict

- **Signal — Weak.** The excess-Sharpe ladder humps on the boundary (+0.602 crossover vs +0.437 broad HY, +0.241 IG) and the crossover premium's sign is positive in every era, every fund pair and every bandwidth. It fails on robustness, not on sign. (i) The headline duration- and equity-adjusted alpha vs broad high yield is **+2.51%/yr at HAC *t* = +2.01** — on the bar, with a bootstrap CI of [+0.38%, +4.74%] whose lower bound is 0.38 pp — and it survives only 5/15 one-year-out deletions (+1.43 without 2016; +1.61 without both wave years). (ii) It is a first-era effect: +4.47%/yr (*t* = +2.63) before 2019 against +1.50%/yr (*t* = +0.92) after, despite the late era containing the record downgrade wave. (iii) Both legs swapped for siblings: +0.63%/yr (*t* = +0.48) — but on that same 2017-2026 window the headline pair itself reads only +0.96%/yr (*t* = +0.67), so this is (ii) restated plus about 1.2 pp of fund-identity noise, not a third independent failure: two pieces of adverse evidence, not three. (iv) The IG side of the boundary — the half that makes it a *rung* — is absent: +1.51%/yr (*t* = +1.15), CI [-0.61%, +3.90%], and -0.05%/yr recently. The synthetic control recovers a planted 3.00%/yr at *t* = +4.64 and fires 0/8 on the null, so the borderline reading is the tape's, not the harness's. *Survivorship: six living ETFs; failed crossover funds are not on this tape. One US credit history, two downgrade waves.*
- **Tradability — Fragile.** Not a cost mirage: expression A is one trade with ≤ 1 bp/yr amortised drag, no borrow, fees inside the tape, and it delivered +0.165 of excess Sharpe. Fragile because the admission price is a 7.3 pp deeper worst drawdown (-22.0% → -29.3%) and 18% more vol, for a premium that is episodic, era-contingent and not measurable at all on the post-2017 tape; and the pure long/short expression decays to *t* = +1.69 at a realistic 50 bps borrow. A tilt, not a harvest.